In [ ]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 330.6/363.4 MB 39.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.2 MB/s eta 0:00:00
ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone m

In [ ]:
!pip install ray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 MB 10.7 MB/s eta 0:00:00


In [ ]:
!pip install supersuit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.3/563.3 kB 28.2 MB/s eta 0:00:00


In [ ]:
!pip install pettingzoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 32.6 MB/s eta 0:00:00


In [ ]:
!pip install pymunk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from supersuit import color_reduction_v0, frame_stack_v1, resize_v1
from torch.distributions.categorical import Categorical

from pettingzoo.butterfly import pistonball_v6

# Определение архитектуры нейросетевого агента
class Agent(nn.Module):
    def __init__(self, num_actions):
        super().__init__()

        # Основная CNN сеть для обработки изображений
        self.network = nn.Sequential(
            self._layer_init(nn.Conv2d(4, 32, 3, padding=1)),  # 1й сверточный слой
            nn.MaxPool2d(2),  # Уменьшение размерности
            nn.ReLU(),  # Функция активации
            self._layer_init(nn.Conv2d(32, 64, 3, padding=1)),  # 2й сверточный слой
            nn.MaxPool2d(2),
            nn.ReLU(),
            self._layer_init(nn.Conv2d(64, 128, 3, padding=1)),  # 3й сверточный слой
            nn.MaxPool2d(2),
            nn.ReLU(),
            nn.Flatten(),  # Преобразование в 1D вектор
            self._layer_init(nn.Linear(128 * 8 * 8, 512)),  # Полносвязный слой
            nn.ReLU(),
        )
        # Выходные слои для актера (политики) и критика (значения)
        self.actor = self._layer_init(nn.Linear(512, num_actions), std=0.01)
        self.critic = self._layer_init(nn.Linear(512, 1))

    # Метод для инициализации слоев
    def _layer_init(self, layer, std=np.sqrt(2), bias_const=0.0):
        torch.nn.init.orthogonal_(layer.weight, std)  # Ортогональная инициализация
        torch.nn.init.constant_(layer.bias, bias_const)  # Инициализация смещений
        return layer

    # Получение значения состояния от критика
    def get_value(self, x):
        return self.critic(self.network(x / 255.0))  # Нормализация входных данных

    # Получение действия и вспомогательных значений
    def get_action_and_value(self, x, action=None):
        hidden = self.network(x / 255.0)  # Проход через сеть
        logits = self.actor(hidden)  # Логиты для вероятностей действий
        probs = Categorical(logits=logits)  # Категориальное распределение

        if action is None:
            action = probs.sample()  # Выбор действия согласно политике

        return action, probs.log_prob(action), probs.entropy(), self.critic(hidden)

# Функция для преобразования наблюдений в батч тензоров
def batchify_obs(obs, device):
    """Конвертирует наблюдения в формат батча тензоров"""
    obs = np.stack([obs[a] for a in obs], axis=0)  # Объединение в массив numpy
    obs = obs.transpose(0, -1, 1, 2)  # Изменение порядка измерений
    obs = torch.tensor(obs).to(device)  # Конвертация в тензор
    return obs

# Функция для преобразования данных среды в батч тензоров
def batchify(x, device):
    """Конвертирует данные среды в батч тензоров"""
    x = np.stack([x[a] for a in x], axis=0)
    x = torch.tensor(x).to(device)
    return x

# Функция для преобразования батча в формат среды
def unbatchify(x, env):
    """Конвертирует батч тензоров в формат среды"""
    x = x.cpu().numpy()
    x = {a: x[i] for i, a in enumerate(env.possible_agents)}
    return x

if __name__ == "__main__":
    """ПАРАМЕТРЫ АЛГОРИТМА"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Выбор устройства
    ent_coef = 0.1  # Коэффициент энтропии
    vf_coef = 0.1  # Коэффициент для функции ценности
    clip_coef = 0.1  # Коэффициент для PPO-clip
    gamma = 0.99  # Коэффициент дисконтирования
    batch_size = 32  # Размер батча
    stack_size = 4  # Количество кадров в стеке
    frame_size = (64, 64)  # Размер кадра
    max_cycles = 500  # Максимальное количество шагов в эпизоде
    total_episodes = 5  # Количество эпизодов обучения

    """НАСТРОЙКА СРЕДЫ"""
    env = pistonball_v6.parallel_env(
        render_mode="rgb_array", continuous=False, max_cycles=max_cycles
    )
    env = color_reduction_v0(env)  # Уменьшение цветового пространства
    env = resize_v1(env, frame_size[0], frame_size[1])  # Изменение размера кадров
    env = frame_stack_v1(env, stack_size=stack_size)  # Накопление кадров

    num_agents = len(env.possible_agents)  # Количество агентов
    num_actions = env.action_space(env.possible_agents[0]).n  # Количество действий
    observation_size = env.observation_space(env.possible_agents[0]).shape  # Размер наблюдения

    """ИНИЦИАЛИЗАЦИЯ АГЕНТА"""
    agent = Agent(num_actions=num_actions).to(device)  # Создание агента
    optimizer = optim.Adam(agent.parameters(), lr=0.001, eps=1e-5)  # Оптимизатор

    """ПОДГОТОВКА ХРАНИЛИЩА ДАННЫХ"""
    end_step = 0
    total_episodic_return = 0
    # Буферы для хранения данных эпизода
    rb_obs = torch.zeros((max_cycles, num_agents, stack_size, *frame_size)).to(device)
    rb_actions = torch.zeros((max_cycles, num_agents)).to(device)
    rb_logprobs = torch.zeros((max_cycles, num_agents)).to(device)
    rb_rewards = torch.zeros((max_cycles, num_agents)).to(device)
    rb_terms = torch.zeros((max_cycles, num_agents)).to(device)
    rb_values = torch.zeros((max_cycles, num_agents)).to(device)

    """ОСНОВНОЙ ЦИКЛ ОБУЧЕНИЯ"""
    for episode in range(total_episodes):
        # Сбор эпизода данных
        with torch.no_grad():
            next_obs, info = env.reset(seed=None)
            total_episodic_return = 0

            for step in range(0, max_cycles):
                obs = batchify_obs(next_obs, device)  # Подготовка наблюдений

                # Получение действий от агента
                actions, logprobs, _, values = agent.get_action_and_value(obs)

                # Шаг в среде
                next_obs, rewards, terms, truncs, infos = env.step(
                    unbatchify(actions, env)

                # Сохранение данных в буфер
                rb_obs[step] = obs
                rb_rewards[step] = batchify(rewards, device)
                rb_terms[step] = batchify(terms, device)
                rb_actions[step] = actions
                rb_logprobs[step] = logprobs
                rb_values[step] = values.flatten()

                total_episodic_return += rb_rewards[step].cpu().numpy()

                # Проверка на завершение эпизода
                if any([terms[a] for a in terms]) or any([truncs[a] for a in truncs]):
                    end_step = step
                    break

        # Расчет преимуществ и возвратов
        with torch.no_grad():
            rb_advantages = torch.zeros_like(rb_rewards).to(device)
            for t in reversed(range(end_step)):
                delta = (
                    rb_rewards[t]
                    + gamma * rb_values[t + 1] * rb_terms[t + 1]
                    - rb_values[t]
                )
                rb_advantages[t] = delta + gamma * gamma * rb_advantages[t + 1]
            rb_returns = rb_advantages + rb_values

        # Подготовка данных для обучения
        b_obs = torch.flatten(rb_obs[:end_step], start_dim=0, end_dim=1)
        b_logprobs = torch.flatten(rb_logprobs[:end_step], start_dim=0, end_dim=1)
        b_actions = torch.flatten(rb_actions[:end_step], start_dim=0, end_dim=1)
        b_returns = torch.flatten(rb_returns[:end_step], start_dim=0, end_dim=1)
        b_values = torch.flatten(rb_values[:end_step], start_dim=0, end_dim=1)
        b_advantages = torch.flatten(rb_advantages[:end_step], start_dim=0, end_dim=1)

        # Оптимизация политики и функции ценности
        b_index = np.arange(len(b_obs))
        clip_fracs = []
        for repeat in range(3):  # Несколько эпох обучения
            np.random.shuffle(b_index)
            for start in range(0, len(b_obs), batch_size):
                end = start + batch_size
                batch_index = b_index[start:end]

                # Получение новых значений для выбранного батча
                _, newlogprob, entropy, value = agent.get_action_and_value(
                    b_obs[batch_index], b_actions.long()[batch_index]
                )

                # Расчет логарифмического отношения вероятностей
                logratio = newlogprob - b_logprobs[batch_index]
                ratio = logratio.exp()

                with torch.no_grad():
                    # Расчет приближенного KL-дивергенции
                    old_approx_kl = (-logratio).mean()
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clip_fracs += [
                        ((ratio - 1.0).abs() > clip_coef).float().mean().item()
                    ]

                # Нормализация преимуществ
                advantages = b_advantages[batch_index]
                advantages = (advantages - advantages.mean()) / (
                    advantages.std() + 1e-8
                )

                # Расчет потерь для политики
                pg_loss1 = -b_advantages[batch_index] * ratio
                pg_loss2 = -b_advantages[batch_index] * torch.clamp(
                    ratio, 1 - clip_coef, 1 + clip_coef
                )
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                # Расчет потерь для функции ценности
                value = value.flatten()
                v_loss_unclipped = (value - b_returns[batch_index]) ** 2
                v_clipped = b_values[batch_index] + torch.clamp(
                    value - b_values[batch_index],
                    -clip_coef,
                    clip_coef,
                )
                v_loss_clipped = (v_clipped - b_returns[batch_index]) ** 2
                v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss = 0.5 * v_loss_max.mean()

                # Общие потери с учетом энтропии
                entropy_loss = entropy.mean()
                loss = pg_loss - ent_coef * entropy_loss + v_loss * vf_coef

                # Шаг оптимизации
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        # Вывод статистики обучения
        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

        print(f"Training episode {episode}")
        print(f"Episodic Return: {np.mean(total_episodic_return)}")
        print(f"Episode Length: {end_step}")
        print("")
        print(f"Value Loss: {v_loss.item()}")
        print(f"Policy Loss: {pg_loss.item()}")
        print(f"Old Approx KL: {old_approx_kl.item()}")
        print(f"Approx KL: {approx_kl.item()}")
        print(f"Clip Fraction: {np.mean(clip_fracs)}")
        print(f"Explained Variance: {explained_var.item()}")
        print("\n-------------------------------------------\n")

    """ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ"""
    env = pistonball_v6.parallel_env(render_mode="human", continuous=False)
    env = color_reduction_v0(env)
    env = resize_v1(env, 64, 64)
    env = frame_stack_v1(env, stack_size=4)

Training episode 0
Episodic Return: -47.23432159423828
Episode Length: 499

Value Loss: 1856.074462890625
Policy Loss: 58.784339904785156
Old Approx KL: -0.0010062797227874398
Approx KL: 5.8731864555738866e-05
Clip Fraction: 0.04023103632478633
Explained Variance: -0.0068128108978271484

-------------------------------------------

Training episode 1
Episodic Return: -50.591697692871094
Episode Length: 499

Value Loss: 11.439495086669922
Policy Loss: -1.5537946224212646
Old Approx KL: 0.002276910236105323
Approx KL: 3.6329030990600586e-05
Clip Fraction: 0.0009348290598290599
Explained Variance: 0.0007278323173522949

-------------------------------------------

Training episode 2
Episodic Return: -52.14876174926758
Episode Length: 499

Value Loss: 5.328505516052246
Policy Loss: 3.1935389041900635
Old Approx KL: -0.002319979015737772
Approx KL: 6.248270074138418e-05
Clip Fraction: 0.006281478939434657
Explained Variance: -0.016857504844665527

-------------------------------------------

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import imageio
from pettingzoo.butterfly import pistonball_v6
from pettingzoo.utils import color_reduction_v0, resize_v1, frame_stack_v1
import torch

# Функция для отображения видео
def show_video(video_path):
    video_file = open(video_path, "rb").read()
    video_url = f"data:video/mp4;base64,{b64encode(video_file).decode('utf-8')}"
    return HTML(f"""<video width=600 controls><source src="{video_url}" type="video/mp4"></video>""")

# Функция для записи видео работы агента в среде
def record_video(env, agent, filename="pistonball.mp4", max_cycles=100):
    # Инициализация среды с настройками для записи видео
    env = pistonball_v6.parallel_env(render_mode="rgb_array", continuous=False, max_cycles=max_cycles)
    # Предобработка наблюдений:
    env = color_reduction_v0(env)  # Уменьшение цветового пространства
    env = resize_v1(env, 64, 64)  # Изменение размера кадров
    env = frame_stack_v1(env, stack_size=4)  # Накопление кадров

    frames = []  # Список для хранения кадров видео
    obs, _ = env.reset()  # Сброс среды

    # Цикл по шагам эпизода
    for _ in range(max_cycles):
        # Преобразование наблюдений в формат для нейросети
        obs_tensor = batchify_obs(obs, device)

        # Получение действий от агента (без вычисления градиентов)
        with torch.no_grad():
            actions, _, _, _ = agent.get_action_and_value(obs_tensor)

        # Преобразование действий в формат среды
        actions = unbatchify(actions, env)

        # Шаг в среде
        obs, _, terms, truncs, _ = env.step(actions)

        # Получение текущего кадра для видео
        frame = env.render()
        frames.append(frame)

        # Проверка на завершение эпизода
        if any(terms.values()) or any(truncs.values()):
            break

    env.close()  # Закрытие среды

    # Сохранение кадров в видеофайл
    imageio.mimsave(filename, frames, fps=20)
    return filename

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Запись и отображение видео после обучения
print("Recording video...")
video_path = record_video(env, agent)
print("Training completed! Showing result:")
show_video(video_path)

In [ ]:
# Функция для отображения видео с обработкой ошибок
def show_video(video_path):
    try:
        video_file = open(video_path, "rb").read()

        video_url = f"data:video/mp4;base64,{b64encode(video_file).decode('utf-8')}"

        display(HTML(f"""<video width=600 controls><source src="{video_url}" type="video/mp4"></video>"""))

        display(Video(video_path, embed=True))
    except Exception as e:
        print(f"Ошибка при отображении видео: {e}")

# Функция для записи видео работы агента в среде
def record_video(env, agent, filename="pistonball.mp4", max_cycles=100):
    # Инициализация среды с параметрами для записи видео:
    env = pistonball_v6.parallel_env(render_mode="rgb_array", continuous=False, max_cycles=max_cycles)

    # Предварительная обработка изображений:
    env = color_reduction_v0(env)  # Уменьшение цветового пространства
    env = resize_v1(env, 64, 64)   # Изменение размера кадров
    env = frame_stack_v1(env, stack_size=4)  # Накопление нескольких кадров

    frames = []  # Список для хранения отдельных кадров видео
    obs, _ = env.reset()  # Сброс среды в начальное состояние

    for _ in range(max_cycles):
        # Преобразование наблюдений в формат тензора для нейросети
        obs_tensor = batchify_obs(obs, device)

        # Получение действий от агента (без вычисления градиентов)
        with torch.no_grad():
            actions, _, _, _ = agent.get_action_and_value(obs_tensor)

        # Преобразование действий в формат, понятный среде
        actions = unbatchify(actions, env)

        # Выполнение действия в среде
        obs, _, terms, truncs, _ = env.step(actions)

        # Получение текущего кадра и добавление его в список
        frame = env.render()
        frames.append(frame)

        # Проверка условий завершения эпизода
        if any(terms.values()) or any(truncs.values()):
            break

    env.close()  # Закрытие среды после записи

    # Сохранение всех кадров в видеофайл с частотой 20 кадров/сек
    imageio.mimsave(filename, frames, fps=20)
    return filename

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Записываем видео работы агента
video_path = "pistonball.mp4"
print("Recording video...")
video_path = record_video(env, agent, video_path)  # Запись видео

# Проверяем существование и размер файла через командную строку
!ls -lh {video_path}

# Выводим записанное видео
print("Displaying video...")
show_video(video_path)

Recording video...
-rw-r--r-- 1 root root 40K Mar 30 14:17 pistonball.mp4
Displaying video...
